# PDF Structure Exploration — Key Findings

This notebook analyzes the internal structure of the **Power SCADA Operation 2020 System Guide** before moving into text cleaning, chunking, embeddings, or RAG.

---

## 1. Document Structure

The PDF is highly structured.

Key numbers:

- Total pages: **1,314**
- Built-in Table of Contents entries: **1,224**
- TOC depth: **Level 1 to Level 7**

This means the document has a reliable chapter, section, subsection, and topic hierarchy.

The built-in TOC should be used as the backbone for:

- Metadata
- Retrieval
- Citations
- TOC-aware chunking
- Source traceability

Example citation style:

```text
Configure > Web Applications > TGML snippets — Page 500
```

---

## 2. Text Extraction Quality

The PDF contains extractable text on every page.

Key result:

- Empty pages: **0**

This means OCR is not required for the core text pipeline.

Images should be treated as supporting visual references. OCR can remain an optional future enhancement only for selected images if needed.

---

## 3. Header and Footer Patterns

The extracted page text contains repeated mechanical lines such as:

```text
System Guide
Page X of 1314
7EN02-0440-03
```

These lines should be removed during light text cleaning because they add noise to retrieval and embeddings.

However, chapter-level labels should not be aggressively removed, such as:

```text
Configure
Plan
Operate
Reference
Install and upgrade
```

These labels provide useful document context and can be preserved as metadata.

---

## 4. Page Text Length Distribution

The page text length distribution gives a useful hint for chunking.

```text
0 empty pages:        0
1–500 characters:    203 pages
501–1500 characters: 509 pages
1501–3000 characters: 599 pages
3000+ characters:    3 pages
```

Most pages contain a moderate amount of text, which makes the document suitable for section-aware chunking.

---

## 5. Chunking Strategy

The recommended approach is not blind fixed-size splitting.

The best strategy is:

```text
TOC-aware chunking
Section-aware chunking
```

Initial chunking target:

```text
Target chunk size: 2,000–3,000 characters
Overlap: only when splitting long sections
```

The chunking process should:

- Use the TOC hierarchy to define section boundaries
- Preserve page references
- Preserve section paths
- Keep short sections together when appropriate
- Split long sections with limited overlap
- Attach rich metadata to every chunk

---

## 6. Heading Detection Notes

A regex-based heading scan detected **1,226 heading candidates**, but the result is noisy.

Many detected candidates are not true section headings, such as:

```text
WARNING
NOTICE
CONFIDENTIALITY
```

Some detected candidates are also numbered procedure steps rather than headings.

Therefore, regex-based heading detection should not be used as the primary structure source.

The built-in TOC is more reliable and should be the main source for document hierarchy.

---

## 7. Final Decision

The PDF is highly suitable for a high-quality RAG pipeline because it has:

- A strong built-in TOC
- Extractable text on every page
- Clear section hierarchy
- Strong potential for page-level and section-level citations
- Supporting images that can be linked later as visual references

The next stage is:

```text
03_text_extraction_and_cleaning.ipynb
```

This stage will extract page-level text and apply light cleaning before TOC-aware chunking.

In [2]:
import sys
import json
import re
from pathlib import Path
from collections import Counter

import fitz
from IPython.display import display, Markdown

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.settings import (
    PDF_PATH,
    MANIFESTS_DIR,
    PDF_STRUCTURE_PATH,
    OUTPUT_DIRS,
)

for d in OUTPUT_DIRS:
    d.mkdir(parents=True, exist_ok=True)

In [4]:
doc = fitz.open(str(PDF_PATH))
toc = doc.get_toc(simple=True)  # list of [level, title, page]

if toc:
    display(Markdown(f"### Table of Contents found: {len(toc)} entries"))

    # Show first 30 entries
    rows = "\n".join(
        f"| {level} | {title.strip()} | {page} |"
        for level, title, page in toc[:30]
    )
    display(Markdown(f"""
| Level | Title | Page |
|-------|-------|------|
{rows}

*Showing first 30 of {len(toc)} entries.*
""".strip()))

    # Level distribution
    level_counts = Counter(level for level, _, _ in toc)
    level_rows = "\n".join(f"| {lvl} | {cnt} |" for lvl, cnt in sorted(level_counts.items()))
    display(Markdown(f"""
#### TOC Level Distribution

| Level | Count |
|-------|-------|
{level_rows}
""".strip()))
else:
    display(Markdown("### No Table of Contents found in PDF."))

### Table of Contents found: 1224 entries

| Level | Title | Page |
|-------|-------|------|
| 1 | Contents | 4 |
| 1 | Safety Precautions | 33 |
| 1 | Introduction | 36 |
| 2 | How this guide is organized | 36 |
| 2 | Content updates | 36 |
| 2 | Assumptions | 37 |
| 2 | What's new | 37 |
| 2 | Resources | 39 |
| 1 | Cybersecurity | 42 |
| 2 | Cybersecurity awareness | 42 |
| 2 | Cybersecurity features | 42 |
| 2 | Recommended actions | 43 |
| 1 | Plan | 44 |
| 2 | Overview | 45 |
| 2 | Components and single-site architectures | 47 |
| 3 | Components overview | 48 |
| 3 | Time synchronization | 48 |
| 3 | Power SCADA Server component | 49 |
| 3 | Server component architecture | 50 |
| 4 | Native architectural redundancy | 50 |
| 4 | Making changes while online | 51 |
| 4 | Ethernet network redundancy | 51 |
| 3 | Power SCADA Control Client component (HTML5 client) | 52 |
| 3 | Control Client component architecture | 53 |
| 3 | Power SCADA View-only Client component (thick client) | 55 |
| 3 | Power SCADA View-only Client architectures | 56 |
| 3 | Event Notification Module component | 56 |
| 3 | Advanced Reporting and Dashboards component | 58 |
| 4 | Advanced Reporting and Dashboards architectures | 58 |
| 4 | Additional Advanced Reporting Modules component | 61 |

*Showing first 30 of 1224 entries.*

#### TOC Level Distribution

| Level | Count |
|-------|-------|
| 1 | 13 |
| 2 | 70 |
| 3 | 256 |
| 4 | 345 |
| 5 | 346 |
| 6 | 192 |
| 7 | 2 |

In [5]:
page_stats = []

for i in range(len(doc)):
    page = doc[i]
    text = page.get_text("text")
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    page_stats.append({
        "page": i + 1,
        "char_count": len(text),
        "line_count": len(lines),
        "first_line": lines[0] if lines else "",
        "last_line": lines[-1] if lines else "",
        "is_empty": len(text.strip()) == 0,
    })

total = len(page_stats)
empty_pages = [p for p in page_stats if p["is_empty"]]
avg_chars = sum(p["char_count"] for p in page_stats) / total if total else 0
avg_lines = sum(p["line_count"] for p in page_stats) / total if total else 0

display(Markdown(f"""
### Page-Level Text Overview

| Item | Value |
|------|-------|
| **Total pages** | {total} |
| **Empty pages (no text)** | {len(empty_pages)} |
| **Avg characters/page** | {avg_chars:.0f} |
| **Avg lines/page** | {avg_lines:.0f} |
""".strip()))

if empty_pages:
    example_rows = ", ".join(str(p["page"]) for p in empty_pages[:20])
    display(Markdown(f"**Empty pages (up to 20):** {example_rows}"))

### Page-Level Text Overview

| Item | Value |
|------|-------|
| **Total pages** | 1314 |
| **Empty pages (no text)** | 0 |
| **Avg characters/page** | 1384 |
| **Avg lines/page** | 36 |

In [6]:
first_lines = Counter(p["first_line"] for p in page_stats if p["first_line"])
last_lines = Counter(p["last_line"] for p in page_stats if p["last_line"])

def show_top_patterns(counter, label, n=15):
    top = counter.most_common(n)
    rows = "\n".join(
        f"| `{text[:80]}` | {count} |" for text, count in top
    )
    display(Markdown(f"""
#### {label} (top {n})

| Text | Occurrences |
|------|-------------|
{rows}
""".strip()))

show_top_patterns(first_lines, "Most Common First Lines")
show_top_patterns(last_lines, "Most Common Last Lines")

#### Most Common First Lines (top 15)

| Text | Occurrences |
|------|-------------|
| `Configure` | 247 |
| `Reference` | 241 |
| `Operate` | 50 |
| `Plan` | 38 |
| `Install and upgrade` | 37 |
| `Logic Code` | 16 |
| `Contents` | 15 |
| `Attribute` | 14 |
| `Troubleshoot and FAQs` | 12 |
| `Applications` | 8 |
| `WARNING` | 6 |
| `TGML Element` | 6 |
| `System Guide` | 5 |
| `Administer` | 5 |
| `Field Name` | 4 |

#### Most Common Last Lines (top 15)

| Text | Occurrences |
|------|-------------|
| `Animatable: Yes` | 6 |
| `https://ipaddress/webhmi)` | 5 |
| `The following screen is displayed.` | 4 |
| `section.` | 4 |
| `For example:` | 3 |
| `considered accurate.` | 3 |
| `accurate.` | 3 |
| `</TGML>` | 3 |
| `Example on screen:` | 3 |
| `connection between OFS and the time stamping module.` | 2 |
| `chronize the ERT module’s time clock.` | 2 |
| `as well.` | 2 |
| `damage, or permanent loss of data.` | 2 |
| `4. At the bottom of the View Library, click Add Folder:` | 2 |
| `5. Enter the folder Name:` | 2 |

In [7]:
heading_pattern = re.compile(
    r"^(?:"
    r"\d+[\.\d]*\s+[A-Z]"        # "1.2 Configuration..."
    r"|Chapter\s+\d+"             # "Chapter 3"
    r"|CHAPTER\s+\d+"             # "CHAPTER 3"
    r"|[A-Z][A-Z\s]{4,}$"        # "ALL CAPS LINE"
    r")",
    re.MULTILINE,
)

heading_samples = []

for i in range(len(doc)):
    page = doc[i]
    text = page.get_text("text")
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    for line in lines[:10]:  # only check first 10 lines per page
        if heading_pattern.match(line):
            heading_samples.append({"page": i + 1, "heading": line[:100]})

display(Markdown(f"### Heading Candidates Found: {len(heading_samples)}"))

# Show first 25
rows = "\n".join(
    f"| {h['page']} | `{h['heading']}` |" for h in heading_samples[:25]
)
display(Markdown(f"""
| Page | Heading |
|------|---------|
{rows}

*Showing first 25 of {len(heading_samples)} candidates.*
""".strip()))

### Heading Candidates Found: 1226

| Page | Heading |
|------|---------|
| 33 | `WARNING` |
| 33 | `UNINTENDED EQUIPMENT OPERATION` |
| 37 | `NOTICE` |
| 37 | `INOPERABLE SYSTEM` |
| 42 | `WARNING` |
| 42 | `CONFIDENTIALITY` |
| 46 | `NOTICE` |
| 46 | `INOPERABLE SYSTEM` |
| 81 | `2. In the left pane, click Common Platform > System Management Server. The following is` |
| 81 | `3. Select This machine is the System Management Server. Review the notes on the screen` |
| 81 | `4. Click Configure. If an existing binding is found for the specified ports, the following message` |
| 81 | `5. Click Yes if you wish to replace the binding. The Configurator will start configuring the Sys-` |
| 82 | `6. On successful configuration, the message “Device configuration completed” is displayed.` |
| 82 | `7. If the configuration is unsuccessful, check the ArchestrA Logger. You can access this by typ-` |
| 94 | `1. Power SCADA acquires historical (trend) data from all devices.` |
| 94 | `2. The Extract-Transform-Load (ETL) tool transfers historical data from Power` |
| 111 | `1.   Synchronize the time` |
| 111 | `2. Time stamping of events` |
| 118 | `1. Events are detected and time stamped by the time stamping module` |
| 118 | `2. Manage the time stamping events using OFS` |
| 118 | `3. Transfer these events to SCADA using OFS, and display them on the SCADA pages` |
| 121 | `WARNING` |
| 122 | `WARNING` |
| 122 | `CONFIDENTIALITY` |
| 126 | `1. Insert the Power SCADA Operation with Advanced Reporting and Dashboards DVD into the` |

*Showing first 25 of 1226 candidates.*

In [8]:
# Bucket pages by character count
buckets = {"0 (empty)": 0, "1-500": 0, "501-1500": 0, "1501-3000": 0, "3000+": 0}

for p in page_stats:
    c = p["char_count"]
    if c == 0:
        buckets["0 (empty)"] += 1
    elif c <= 500:
        buckets["1-500"] += 1
    elif c <= 1500:
        buckets["501-1500"] += 1
    elif c <= 3000:
        buckets["1501-3000"] += 1
    else:
        buckets["3000+"] += 1

rows = "\n".join(f"| {k} | {v} |" for k, v in buckets.items())
display(Markdown(f"""
### Page Text Length Distribution

| Characters | Pages |
|------------|-------|
{rows}
""".strip()))

### Page Text Length Distribution

| Characters | Pages |
|------------|-------|
| 0 (empty) | 0 |
| 1-500 | 203 |
| 501-1500 | 509 |
| 1501-3000 | 599 |
| 3000+ | 3 |

In [10]:
sample_pages = [1, 2, 10, 50, 100, 500, 1000]
sample_pages = [p for p in sample_pages if p <= len(doc)]

for pg in sample_pages:
    page = doc[pg - 1]
    text = page.get_text("text").strip()
    preview = text[:600] + ("..." if len(text) > 600 else "")
    display(Markdown(f"""
---
#### Page {pg} ({len(text)} chars)

```
{preview}
```
""".strip()))

---
#### Page 1 (113 chars)

```
EcoStruxure™
Power SCADA Operation 2020 with Advanced
Reporting and Dashboards
System Guide
7EN02-0440-03
04/2022
```

---
#### Page 2 (1473 chars)

```
Legal Information
The Schneider Electric brand and any registered trademarks of Schneider Electric Industries SAS
referred to in this guide are the sole property of Schneider Electric SA and its subsidiaries. They
may not be used for any purpose without the owner's permission, given in writing. This guide and
its content are protected, within the meaning of the French intellectual property code (Code de la
propriété intellectuelle français, referred to hereafter as "the Code"), under the laws of copyright
covering texts, drawings and models, as well as by trademark law. You agree not to reprod...
```

---
#### Page 10 (1404 chars)

```
Contents
System Guide
Page 10 of 1314
7EN02-0440-03
Enable Waveforms
256
Enabling waveforms for onboard alarms
257
Adding an onboard alarm tag
258
Managing device profiles
258
Adding device profiles
258
Editing device profiles
259
Deleting device profiles
260
IEC 61850 system setup workflow
260
Create IEC 61850 Device Type
261
Managing IEC 61850 datasets
263
Edit IEC 61850 Report control blocks
264
Edit driver parameters
266
Set Up Trend Intervals
266
Select Trend Intervals
266
Trend tag scan intervals
267
Disk storage calculation for trends
268
Create composite device profiles
268
Creating a ...
```

---
#### Page 50 (2093 chars)

```
Plan
System Guide
Page 50 of 1314
7EN02-0440-03
• I/O point count is now tag based not address based. For example, two tags that use the same
PLC address will be counted twice. If two trend tags use the same variable tag, it will be coun-
ted once. The same applies to alarms.
• For the multi-process mode, each server component will accumulate its own point count. The
server component point count is the count added up from all server components. If two server
components use the same tags, say alarm and trend, the tags will be counted twice when the
point count gets summed.
• For the multi-proce...
```

---
#### Page 100 (1089 chars)

```
Plan
System Guide
Page 100 of 1314
7EN02-0440-03
EcoStruxure Web Services (EWS)
EcoStruxure Web Services (EWS) for Power SCADA Operation shares real-time, historical, and
alarm data with EcoStruxure™Building Operations (EBO) and historical data with Power
Monitoring Expert (PME). Do not confuse this feature with the EWS Server that was released as a
part of PowerSCADA Expert/Vijeo Citect version 7.40 (which is for tag level process data).
EWS uses web-based HTTP protocol to transfer data. It enables two-way data transfers, which
allows the acknowledgment of alarms from EBO. To include this new...
```

---
#### Page 500 (828 chars)

```
Configure
System Guide
Page 500 of 1314
7EN02-0440-03
Snippet type
Syntax
Attributes Description
and password.
Optional Attributes for snippet
Type User credential PopUp
ShowTitleBar: Displays the title
bar in the target pane when set to
Yes.
Adding a diagram to the menu bar
You can add a diagram to the menu bar and then use it to navigate to diagrams.
This topic uses an example to demonstrate how to accomplish this.
To add a diagram to the menu bar:
1. Log in to PSO Web Applications (https//localhost/webhmI or ipaddress/webhmi).
The following screen is displayed.
2. Example: A user wants to d...
```

---
#### Page 1000 (2057 chars)

```
Reference
System Guide
Page 1000 of 1314
7EN02-0440-03
5. Click Initialize. Mapped Trend logs appear with associated timestamp data for each.
You should see a row for each pair selected in the Mappings tab. The Key is a long string
that represents the pair.
Now, the next time you run ETL, only data after the given timestamp is loaded.
6. Run the ETL job.
The Target Device value is the PME source under which the PSO data will be loaded.
7. (Optional) Verify the data transfer. See "Verifying PSO data transfer to PME" on page 1000
for details.
Verifying PSO data transfer to PME
After a PSO to PME...
```

In [11]:
structure_data = {
    "total_pages": len(doc),
    "toc_entries": len(toc),
    "toc": [{"level": l, "title": t, "page": p} for l, t, p in toc],
    "empty_pages": [p["page"] for p in page_stats if p["is_empty"]],
    "page_stats": page_stats,
    "heading_candidates_count": len(heading_samples),
}

PDF_STRUCTURE_PATH.write_text(json.dumps(structure_data, indent=2), encoding="utf-8")
doc.close()

display(Markdown(f"""
### Structure Data Saved

| Item | Value |
|------|-------|
| **TOC entries** | {structure_data['toc_entries']} |
| **Empty pages** | {len(structure_data['empty_pages'])} |
| **Heading candidates** | {structure_data['heading_candidates_count']} |
| **Saved to** | `{PDF_STRUCTURE_PATH.relative_to(project_root)}` |
""".strip()))

### Structure Data Saved

| Item | Value |
|------|-------|
| **TOC entries** | 1224 |
| **Empty pages** | 0 |
| **Heading candidates** | 1226 |
| **Saved to** | `SCADA-DIP\data\manifests\pdf_structure.json` |

---